# Gold Layer — Business Insight Views
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | `gbmart.gold.fact_sales` + dimensions |
| **Target** | `gbmart.gold.vw_*` (views, not tables) |
| **Consumers** | Downstream BI/analytics teams — run **daily**, pre-aggregated off `fact_sales` |

> These are real, recurring business questions — not one-off ad-hoc
> queries. Every day, a category manager wants to know "how did each
> category do this month," and a regional ops lead wants to know "how is
> each city/state performing." Without a view, both teams would write
> their own `fact_sales` joins — and likely get slightly different
> numbers depending on how each person filters/joins. A view gives
> everyone the same pre-aggregated answer, every day, from one definition.

### Why a VIEW, not a materialized/aggregate table
At `fact_sales`'s current size (~377K rows), a `VIEW` recomputes fresh on
every query — no separate refresh job to schedule or monitor, and it
always reflects whatever's currently in `fact_sales`. If a dashboard ever
gets slow at much larger scale, that's the point where you'd promote one
to a materialized aggregate table — not before.

## Step 1 — Setup

In [0]:
CATALOG = "gbmart"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

## View 1 — `vw_monthly_category_sales`

### Business use case
**Who runs this daily:** Category/Merchandising managers.
**Question it answers:** "How is each product category trending, month
over month — and which categories are leaning on discounts to move
volume?"

This is exactly the kind of view a merchandising team checks **every
morning** — yesterday's run rolls forward into the current month's
running total the moment new orders land in `fact_sales`. Without this
view, every analyst re-derives "revenue by category by month" slightly
differently (different discount handling, different date truncation) —
this view is the one agreed-upon definition everyone reports against.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gbmart.gold.vw_monthly_category_sales AS
SELECT
    d.year,
    d.month,
    p.category,
    p.sub_category,
    SUM(f.Quantity_purchased)                          AS total_quantity_sold,
    SUM(f.Sales_amount)                                AS total_revenue,
    COUNT(DISTINCT f.Order_ID)                         AS total_orders,
    ROUND(AVG(f.Actual_price - f.Discounted_price), 2) AS avg_discount_given
FROM gbmart.gold.fact_sales f
JOIN gbmart.gold.dim_product p ON f.Product_ID = p.product_id AND p.is_current = true
JOIN gbmart.gold.dim_date d    ON f.Time_ID = d.date_key
GROUP BY d.year, d.month, p.category, p.sub_category
""")
print("vw_monthly_category_sales created")

### Insight: top categories by revenue, this year

In [0]:
spark.sql("""
    SELECT category, SUM(total_revenue) AS yearly_revenue, SUM(total_orders) AS yearly_orders
    FROM gbmart.gold.vw_monthly_category_sales
    GROUP BY category
    ORDER BY yearly_revenue DESC
""").display()

## View 2 — `vw_regional_sales`

### Business use case
**Who runs this daily:** Regional/logistics ops leads.
**Question it answers:** "Which states/cities are driving the most
revenue and orders, and how big is the average order in each region?"

Regional ops teams use this to decide where to prioritize warehouse
stock, run local promotions, or flag underperforming markets. A region
with high `total_orders` but low `avg_order_value` tells a very different
story than one with few orders but a high average — same revenue,
different operational response.

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW gbmart.gold.vw_regional_sales AS
SELECT
    a.state,
    a.city,
    COUNT(DISTINCT f.Order_ID)                                 AS total_orders,
    COUNT(DISTINCT f.Customer_ID)                              AS total_customers,
    SUM(f.Quantity_purchased)                                  AS total_quantity_sold,
    SUM(f.Sales_amount)                                        AS total_revenue,
    ROUND(SUM(f.Sales_amount) / COUNT(DISTINCT f.Order_ID), 2) AS avg_order_value
FROM gbmart.gold.fact_sales f
JOIN gbmart.gold.dim_address a ON f.Address_ID = a.address_id
GROUP BY a.state, a.city
""")
print("vw_regional_sales created")

### Insight: top 10 states by revenue

In [0]:
spark.sql("""
    SELECT state, SUM(total_revenue) AS state_revenue, SUM(total_orders) AS state_orders,
           ROUND(SUM(total_revenue) / SUM(total_orders), 2) AS state_avg_order_value
    FROM gbmart.gold.vw_regional_sales
    GROUP BY state
    ORDER BY state_revenue DESC
    LIMIT 10
""").display()

## Reset (if needed)

In [0]:
# spark.sql("DROP VIEW IF EXISTS gbmart.gold.vw_monthly_category_sales")
# spark.sql("DROP VIEW IF EXISTS gbmart.gold.vw_regional_sales")
# print("Reset complete")